# Computational Theory Problems

This notebook works through a series of problems based on the Secure Hash Standard published by NIST as FIPS PUB 180-4. 

## Problem 1: Representing SHA-256 Data

SHA-256 works on fixed-size binary data. Before implementing the algorithm, this section chooses a Python representation for each kind of data it uses, based on the definitions in FIPS 180-4.

The choice matters because Python's built-in `int` has no fixed size. SHA-256 depends on values that are exactly 32 bits wide and that wrap around when they overflow ([FIPS 180-4, Section 3.2](https://doi.org/10.6028/NIST.FIPS.180-4)). The wrong representation produces a wrong hash without raising any error.

For each kind of data, this section:

- states what the standard requires
- explains why the chosen representation is appropiate
- demonstrates the chosen representation with an example




| Data | Size | Representation |
| --- | --- | --- |
| Word | 32 bits | `np.uint32` |
| Sequence of words | $n \times 32$ bits | 1-D NumPy array of `np.uint32` |


Each choice is explained and demonstrated below.

### Imports

[NumPy](https://numpy.org/doc/stable/user/basics.types.html) provides fixed-width integer types such as `uint32`. The standard library module `sys` reports the machine's native byte order.

In [1]:
# Standard library: reports this machine's native byte order.
import sys

# NumPy: provides fixed-width unsigned integers such as uint32.
import numpy as np

### 1. 32-bit Words

FIPS 180-4 ([Section 2.1](https://doi.org/10.6028/NIST.FIPS.180-4)) defines a *word* as a group of $w$ bits, with $w = 32$ for SHA-256. Every SHA-256 operation works on words.

**Choice:** NumPy's `np.uint32`, an unsigned (non-negative) integer that is always exactly 32 bits wide.

**Why it is appropriate:**

- It can't hold more than 32 bits which matches the defintion of a word
- Addition wraps around after $2^{32} - 1$, as required by Section 3.2
- A lift shift removes any bit moved after the 32nd position

Each of these properties is checked below.


In [2]:
# Create a single 32-bit word holding the largest possible value.
word = np.uint32(0xFFFFFFFF)

print(word)
print(type(word))
print(hex(word))

4294967295
<class 'numpy.uint32'>
0xffffffff


**Exactly 32 bits.**

 A 32-bit unsigned integer ranges from $0$ to $2^{32} - 1$. Instead of entering these limits by hand, [`np.iinfo`](https://numpy.org/doc/stable/reference/generated/numpy.iinfo.html) reads them directly from NumPy.

In [3]:
# Show the smallest and largest values a uint32 can hold.
limits = np.iinfo(np.uint32)

print(limits.min, limits.max)
print(hex(limits.max))

0 4294967295
0xffffffff


**Addition wraps modulo $2^{32}$.** 

Adding 1 to the largest `uint32` value gives 0. The same sum with a Python `int` is included for contrast: it continues to 4294967296, which is not a valid 32-bit word.

An array is used here rather than a single value. Adding two single `np.uint32` values also gives 0, but NumPy prints an overflow warning as well. An array wraps without the warning.

In [4]:
# A uint32 array wraps round to 0 when it overflows.
words = np.array([0xFFFFFFFF], dtype=np.uint32)
print(words + np.uint32(1))

# A Python int keeps growing past 32 bits.
print(0xFFFFFFFF + 1)

[0]
4294967296


**Shifting stays within 32 bits.** 

The value `0x80000001` has its highest and lowest bits set. Shifting it left by one moves the highest bit outside the 32-bit range. A `uint32` removes that bit and gives `0x2`, while a Python `int` keeps it and becomes 33 bits long.

In [5]:
# A value with its highest and lowest bits set.
x = np.uint32(0x80000001)

# Shift left by one: the highest bit falls off a uint32.
print(hex(x << 1))

# The same shift on a Python int keeps the extra bit.
print(hex(0x80000001 << 1))

0x2
0x100000002


### 2. Sequences of 32-bit Words

SHA-256 uses several fixed-length sequences of words

8 words for the hash value ([FIPS 180-4, Section 5.3.3](https://doi.org/10.6028/NIST.FIPS.180-4)), 
16 for each message block (Section 5.2.1), and 64 each for the round constants (Section 4.2.2) and the message schedule (Section 6.2.2).

**Choice:** a one-dimensional [NumPy array](https://numpy.org/doc/stable/reference/arrays.ndarray.html) with `dtype=np.uint32`.

**Why it is appropriate:**

- Every element is stored as a `uint32`, so each one keeps the 32-bit behaviour from Section 1
- An operation applied to the whole array is applied to every element at once and each result still wraps at 32 bits
- The words are kept in order and can be accessed by position which matches the subscript notation in the standard like $H_0$ to $H_7$

**Fixed type and size.** 

The eight initial hash values $H^{(0)}$ are stored below. The array reports its type, its length and its total size in bytes. Eight words of 4 bytes each give 32 bytes which is 256 bits.

In [6]:
# The eight initial hash values H(0) from FIPS 180-4, Section 5.3.3.
initial_hash = np.array(
    [0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
     0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19],
    dtype=np.uint32,
)

print(initial_hash.dtype, initial_hash.shape, initial_hash.nbytes)

# A single element keeps the uint32 type.
print(type(initial_hash[0]))

uint32 (8,) 32
<class 'numpy.uint32'>


**Operations stay within 32 bits.**

 Here the array is added to itself so every value gets doubled in one go. Each answer wraps around at $2^{32}$ so they all stay as proper 32-bit words. The same values in a normal Python `list` are shown underneath for comparason. Four of them go over 32 bits, which you can see from the extra `1` at the front.

In [7]:
# Every element is added at once, and each sum wraps at 2**32.
print([hex(word) for word in initial_hash + initial_hash])

# The same values in a Python list are plain ints and do not wrap.
hash_list = [0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
             0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19]
print([hex(word + word) for word in hash_list])

['0xd413ccce', '0x76cf5d0a', '0x78dde6e4', '0x4a9fea74', '0xa21ca4fe', '0x360ad118', '0x3f07b356', '0xb7c19a32']
['0xd413ccce', '0x176cf5d0a', '0x78dde6e4', '0x14a9fea74', '0xa21ca4fe', '0x1360ad118', '0x3f07b356', '0xb7c19a32']
